# Niveaux de risque sur tout le portefeuille : version finale et essais, sur la même grille

**But** : présenter, sur la **même grille de lecture**, les niveaux de risque de trois approches, chacune sur **tous ses clients** (et non sur un échantillon), puis sur les **clients communs aux trois** :
1. **version finale du ML** (`ml_14`) : règle du contentieux, puis CatBoost de `ml_14` pour les autres clients (`evaluation_finale_test.ipynb`, section 6 : train en hors pli, test par le modèle final) ;
2. **séparation du contentieux, jeu corrigé** (`essai_jeu_corrige_deux_populations.ipynb`) : règle du contentieux, puis le modèle hors contentieux (23 colonnes d'origine) ;
3. **séparation par la codification de la banque, jeu de base** (`essai_jeu_de_base_deux_populations.ipynb`) : `PAY_1` ≥ 2 en défaut, puis le modèle pour les autres.

Pour les deux essais, deux versions : le modèle sur les 23 colonnes, et le modèle **sans les `PAY_n`** (le comportement seul).

**Niveaux** : 1. la règle (contentieux, ou `PAY_1` ≥ 2) ; 2. **haut risque** : au-dessus du seuil qui détecte 30 % des défauts des autres clients ; 3. **risque modéré** : jusqu'au seuil qui en détecte 60 % ; 4. **risque faible** : le reste. Pour `ml_14`, les seuils sont ceux du train (évaluation finale) ; pour les essais, ils sont calculés sur leurs probabilités hors pli. Tableaux **descriptifs** : chaque client est classé par un modèle qui ne l'a pas vu, mais seule l'évaluation finale (section 5) est une mesure sur un test jamais utilisé.

**Ce notebook ne réentraîne rien** : il lit les fichiers enregistrés par les trois notebooks (`data/ML/comparatif_global/`). Les périmètres diffèrent (périmètre du ML, niveau 5, fichier brut) : d'où la seconde partie, sur les clients communs.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
dossier = root_dir / "data" / "ML" / "comparatif_global"

def seuil_pour_rappel(y, p, rappel):
    proba_defauts = np.sort(p[y == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

def niveaux_essai(fichier, colonne_regle, prefixe_strategie):
    # Niveaux d'un essai : la règle, puis les notes du modèle (probabilités hors pli) pour les autres clients, bornes à 30 % et 60 % de rappel
    d = pd.read_csv(dossier / fichier)
    col = next(c for c in d.columns if c.startswith(f"note | {prefixe_strategie}"))
    regle = d[colonne_regle] == 1
    y_autres, p_autres = d.loc[~regle, "dpnm"].to_numpy(), d.loc[~regle, col].to_numpy()
    s30, s60 = seuil_pour_rappel(y_autres, p_autres, 0.30), seuil_pour_rappel(y_autres, p_autres, 0.60)
    niveau = np.where(regle, "1. Règle", np.where(d[col] >= s30, "2. Haut risque", np.where(d[col] >= s60, "3. Risque modéré", "4. Risque faible")))
    return pd.DataFrame({"ID": d["ID"], "dpnm": d["dpnm"], "niveau": niveau}), col.removeprefix("note | ")

SOURCES = {}
f14 = dossier / "ml_14_global.csv"
if f14.exists():
    g = pd.read_csv(f14)
    SOURCES["Version finale ml_14 (règle du contentieux + CatBoost de ml_14)"] = pd.DataFrame(
        {"ID": g["ID"], "dpnm": g["dpnm"], "niveau": g["niveau"].str.replace("1. Contentieux (règle)", "1. Règle", regex=False)})
else:
    display(Markdown("⚠️ `ml_14_global.csv` absent : exécuter la section 6 de `evaluation_finale_test.ipynb`."))
for fichier, colonne, libelle in (("jeu_corrige_deux_populations.csv", "contentieux", "Contentieux (définition 2), jeu corrigé"),
                                  ("jeu_de_base_deux_populations.csv", "pay1_retard", "Codification de la banque (PAY_1 ≥ 2), jeu de base")):
    if not (dossier / fichier).exists():
        display(Markdown(f"⚠️ `{fichier}` absent : lancer le notebook en deux populations correspondant."))
        continue
    for prefixe, version in (("2.", "23 colonnes"), ("3.", "sans les PAY_n")):
        niveaux, strategie = niveaux_essai(fichier, colonne, prefixe)
        SOURCES[f"{libelle}, ML {version}"] = niveaux

def tableau_niveaux(d):
    t = d.groupby("niveau")["dpnm"].agg(Clients="size", Défauts="sum")
    t["Précision du niveau seul (taux de défaut, %)"] = (t["Défauts"] / t["Clients"] * 100).round(1)
    t["Part de tous les défauts (%)"] = (t["Défauts"] / t["Défauts"].sum() * 100).round(1)
    t["Défauts cumulés (%)"] = (t["Défauts"].cumsum() / t["Défauts"].sum() * 100).round(1)
    t["Précision cumulée (%)"] = (t["Défauts"].cumsum() / t["Clients"].cumsum() * 100).round(1)
    t["Part des clients (%)"] = (t["Clients"] / t["Clients"].sum() * 100).round(1)
    return t

## 1. Chaque approche sur tous ses clients

Un tableau par approche, sur son propre périmètre : comment se répartissent les clients et les défauts entre les niveaux, et quelle précision on obtient en signalant jusqu'à chaque niveau.

In [ ]:
for nom, d in SOURCES.items():
    display(Markdown(f"**{nom}** ({len(d)} clients, {d['dpnm'].sum()} défauts, taux de défaut {d['dpnm'].mean() * 100:.1f} %)"))
    display(tableau_niveaux(d))

## 2. Sur les clients communs aux trois approches

Mêmes niveaux qu'en partie 1, mais restreints aux clients présents dans les trois fichiers (le périmètre du ML, le plus petit), pour comparer sur exactement les mêmes clients.

In [ ]:
communs = set.intersection(*(set(d["ID"]) for d in SOURCES.values()))
resume = {}
for nom, d in SOURCES.items():
    c = d[d["ID"].isin(communs)]
    t = tableau_niveaux(c)
    resume[nom] = t[["Défauts cumulés (%)", "Précision cumulée (%)", "Part des clients (%)"]]
display(Markdown(f"**Clients communs aux trois approches** : {len(communs)}"))
display(pd.concat(resume, names=["Approche"]))

**Comment lire** : pour chaque approche, les lignes se lisent de haut en bas comme une décision de la banque (« je signale jusqu'à ce niveau ») : part des défauts attrapés et précision obtenue. La comparaison entre approches se fait à niveau égal, sur les clients communs : une approche fait mieux si, au même niveau, elle attrape plus de défauts avec une précision au moins égale.